# Clustering do Open Food Facts

Análise de agrupamento de uma amostra de 300 mil linhas do Open Food Facts. Aplicamos K-Means, DBSCAN e Agrupamento Hierárquico sobre os nutrientes (por 100g) e comparamos os resultados.

Toda a lógica vive no pacote `uaoff` (arquivo único `src/uaoff/__init__.py`). Aqui apenas rodamos a pipeline e mostramos os resultados passo a passo.

## 1. Carregamento

Baixa o dataset do Google Drive (via `gdown`) na primeira vez e usa o cache em `data/` depois.

In [ ]:
from uaoff import load_data

df = load_data()
df.head()

## 2. Pré-processamento

Remove linhas sem nome/categoria, colunas nutricionais pouco preenchidas, spam, duplicatas e outliers. `NaN` é tratado como 0g.

In [ ]:
from uaoff import preprocess

df_limpo, colunas_nutricionais, y_categorias = preprocess(df)
df_limpo.head()

## 3. Normalização + PCA (apenas para visualização)

Normaliza com `StandardScaler` e projeta em 2D com PCA. **O PCA é usado somente nos gráficos**; o agrupamento roda sobre todas as features escaladas (não sobre a projeção). A função imprime a `explained_variance_ratio_` para mostrar quanto de variância é perdida na projeção em 2D.

In [ ]:
from uaoff import reduce_dims

X = df_limpo[colunas_nutricionais].copy()
X_escalado, df_pca = reduce_dims(X)
df_pca['Categoria_Real'] = y_categorias.reset_index(drop=True)
df_pca.head()

## 4. K-Means

Varre K de 2 a 12 e avalia com Silhouette, Davies-Bouldin e Calinski-Harabasz (além do cotovelo) para escolher K.

In [ ]:
from uaoff import kmeans

grupos_kmeans = kmeans(X_escalado, df_pca, df_limpo)

## 5. DBSCAN

Usa `RobustScaler` (mais adequado a dados zero-inflados/assimétricos), o gráfico de k-distâncias para sugerir `eps` e uma busca em grade de `eps` × `min_samples`. O ruído (`-1`) é excluído do cálculo do silhouette/ARI na comparação final.

In [ ]:
from uaoff import dbscan

# passamos os dados brutos X; o dbscan() escala internamente (RobustScaler)
grupos_dbscan = dbscan(X, df_pca, df_limpo)

## 6. Agrupamento Hierárquico

Dendrograma com linkage `ward`, e `K=6` para comparar com o K-Means.

In [ ]:
from uaoff import hierarchical

grupos_hierarquicos = hierarchical(X_escalado, df_pca, df_limpo)

## 7. Comparação

Silhouette Score (coesão) e Adjusted Rand Index (semelhança com a categoria real).

In [ ]:
from uaoff import compare

compare(
    X_escalado,
    y_categorias,
    {
        'K-Means (K=6)': grupos_kmeans,
        'DBSCAN': grupos_dbscan,
        'Hierárquico (K=6)': grupos_hierarquicos,
    },
)

## 8. Perfis dos clusters

Heatmap da média de nutrientes por cluster, com nomes automáticos (ex.: 'high-carb', 'high-fat', 'salty'). O DBSCAN é excluído porque seus rótulos contêm ruído (`-1`).

In [ ]:
from uaoff import cluster_profiles

perfis_kmeans = cluster_profiles(df_limpo, grupos_kmeans, colunas_nutricionais, 'K-Means', 'kmeans_cluster_profiles.png')
perfis_kmeans

In [ ]:
perfis_hierarquico = cluster_profiles(df_limpo, grupos_hierarquicos, colunas_nutricionais, 'Hierárquico', 'hierarchical_cluster_profiles.png')
perfis_hierarquico

## 9. Conclusão

Conclusão escrita (honesta) sobre os achados e limitações.

In [ ]:
from uaoff import conclusion

tabela = compare(
    X_escalado,
    y_categorias,
    {
        'K-Means (K=6)': grupos_kmeans,
        'DBSCAN': grupos_dbscan,
        'Hierárquico (K=6)': grupos_hierarquicos,
    },
)
conclusion(tabela, grupos_dbscan)